# 第14章 scikit-learn 入門

**scikit-learn** は Python の機械学習ライブラリ。
データの **前処理・モデル学習・評価** を一貫した API で行えるのが特徴です。

公式: <https://scikit-learn.org/stable/>

## 学習目標
- scikit-learn の **共通 API** (`fit` / `transform` / `predict`) を理解する
- `train_test_split` でデータを分割できる
- 線形回帰モデルを学習・予測できる
- 評価指標 (MAE / MSE / RMSE / R²) を計算できる


## 14.1 scikit-learn の共通 API

scikit-learn のすばらしいところは、**ほとんどのオブジェクトが同じインターフェース** を持っていることです。

| 種類 | メソッド | 役割 |
|-|-|-|
| 推定器 (Estimator) | `fit(X, y)` | 学習 |
| 推定器 | `predict(X)` | 予測 |
| 変換器 (Transformer) | `fit(X)` → `transform(X)` または `fit_transform(X)` | データ変換 |

「**前処理もモデルも、まず `fit`、それから `transform` か `predict`**」と覚えてください。


## 14.2 簡単な線形回帰の例


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# y = 2x + ノイズ という関係のデータを作る
np.random.seed(0)
X = np.linspace(0, 10, 100).reshape(-1, 1)   # (100, 1) の 2 次元配列
y = 2 * X.flatten() + np.random.randn(100)


In [ ]:
# 1. 学習用 / 検証用に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0,
)
print('train:', X_train.shape, 'test:', X_test.shape)


In [ ]:
# 2. モデルのインスタンス化
model = LinearRegression()

# 3. 学習
model.fit(X_train, y_train)

# 4. 学習結果
print(f'傾き a = {model.coef_[0]:.4f}')
print(f'切片 b = {model.intercept_:.4f}')


In [ ]:
# 5. 予測
y_pred = model.predict(X_test)

# 6. 評価
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE  = {mae:.4f}')
print(f'MSE  = {mse:.4f}')
print(f'RMSE = {rmse:.4f}')
print(f'R^2  = {r2:.4f}')


In [ ]:
# 可視化
import matplotlib.pyplot as plt

plt.scatter(X_test, y_test, label='真値', alpha=0.6)
plt.plot(X_test, y_pred, color='red', label='予測')
plt.legend()
plt.title('線形回帰の結果')
plt.show()


## 14.3 前処理の API

スケーリングやエンコーディングも同じ `fit` / `transform` の流れ。


In [ ]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])

scaler = StandardScaler()
scaler.fit(X)               # 平均と標準偏差を計算
X_scaled = scaler.transform(X)
print('平均=0, 分散=1 に変換された:')
print(X_scaled.flatten())


### 大事なルール: **scaler.fit は train だけで行う**

検証データに `fit` してしまうと、検証データの情報が学習に入り込んでしまいます (リーク)。
**train で `fit_transform`, test で `transform` だけ** が鉄則。


In [ ]:
# OK の例
X_train, X_test = np.array([[1.],[2.],[3.],[4.]]), np.array([[5.],[6.]])
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # ← train だけで fit
X_test_s  = scaler.transform(X_test)         # ← test には transform のみ
print('train scaled:\n', X_train_s)
print('test scaled :\n', X_test_s)


## 14.4 Pipeline で前処理 + モデルを 1 本化

`Pipeline` を使うと、前処理とモデルを 1 つのオブジェクトにまとめられます。


In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression()),
])

pipe.fit(X_train, [10, 20, 30, 40])
print(pipe.predict([[5.0]]))


## 14.5 主要なクラスの一覧

| 用途 | クラス |
|-|-|
| データ分割 | `train_test_split`, `KFold`, `StratifiedKFold` |
| スケーリング | `StandardScaler`, `MinMaxScaler` |
| エンコーディング | `LabelEncoder`, `OneHotEncoder` |
| 回帰モデル | `LinearRegression`, `Ridge`, `Lasso`, `RandomForestRegressor` |
| 分類モデル | `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier` |
| クラスタリング | `KMeans`, `DBSCAN` |
| 評価指標 | `mean_squared_error`, `r2_score`, `accuracy_score`, `f1_score` |


## まとめ

- scikit-learn の API は **fit → transform / predict** で統一
- データは必ず **学習用 / 検証用に分けて** 評価する
- 前処理の `fit` は **train だけ** で行う (リーク防止)
- Pipeline でまとめると、本番運用での扱いがしやすい
